# Lokaverkefni

In [1]:
import os
from pathlib import Path
import requests
import zipfile

DATA_DIR = Path("data")
ZIP_PATH = DATA_DIR / "IGC-News1-22.10.TEI.zip"
EXTRACT_DIR = DATA_DIR / "IGC-News1"

URL = "https://repository.clarin.is/repository/xmlui/bitstream/handle/20.500.12537/236/IGC-News1-22.10.TEI.zip"

DATA_DIR.mkdir(exist_ok=True)

def download_file(url: str, dest: Path):
    if dest.exists():
        print(f"File already exists: {dest}")
        return
    
    print(f"Downloading to {dest} ...")
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(dest, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
    print("Download complete.")

def unzip_file(zip_path: Path, extract_dir: Path):
    if extract_dir.exists() and any(extract_dir.iterdir()):
        print(f"Already extracted: {extract_dir}")
        return
    
    print(f"Extracting {zip_path} to {extract_dir} ...")
    extract_dir.mkdir(exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)
    print("Extraction complete.")

download_file(URL, ZIP_PATH)
unzip_file(ZIP_PATH, EXTRACT_DIR)

Download complete.
Extracting data\IGC-News1-22.10.TEI.zip to data\IGC-News1 ...
Extraction complete.


In [20]:
from pathlib import Path
import xml.etree.ElementTree as ET

def local_name(tag):
    return tag.split("}", 1)[-1] if "}" in tag else tag

def parse_article(article_path: Path):
    tree = ET.parse(article_path)
    root = tree.getroot()

    title = None
    source = None
    article_date = None
    keywords = []
    paragraphs = []

    for elem in root.iter():
        name = local_name(elem.tag)

        if name == "analytic":
            for child in elem:
                child_name = local_name(child.tag)
                if child_name == "title" and child.text and not title:
                    title = child.text.strip()
                elif child_name == "date" and child.attrib.get("when") and not article_date:
                    article_date = child.attrib["when"]

        elif name == "monogr":
            for child in elem:
                if local_name(child.tag) == "title" and child.text and not source:
                    source = child.text.strip()

        elif name == "term" and elem.text:
            keywords.append(elem.text.strip())

    # Only keep paragraphs inside the article body
    for body in root.iter():
        if local_name(body.tag) == "body":
            for elem in body.iter():
                if local_name(elem.tag) == "p" and elem.text:
                    text = elem.text.strip()
                    if text:
                        paragraphs.append(text)

    return {
        "path": str(article_path),
        "title": title,
        "source": source,
        "date": article_date,
        "keywords": keywords,
        "text": "\n".join(paragraphs)
    }

In [23]:
from pathlib import Path
import xml.etree.ElementTree as ET

SOURCE = "visir"
YEAR = "2020"

wrapper_path = Path(f"data/IGC-News1/IGC-News1-22.10.TEI/{SOURCE}/IGC-News1-{SOURCE}-22.10.xml")

tree = ET.parse(wrapper_path)
root = tree.getroot()

hrefs = [elem.attrib["href"] for elem in root.iter() if elem.tag.endswith("include")]
hrefs_year = [h for h in hrefs if h.startswith(f"{YEAR}/")]

print("Total article refs:", len(hrefs))
print(f"Article refs in {YEAR}:", len(hrefs_year))
print("First 5:", hrefs_year[:5])

Total article refs: 699957
Article refs in 2020: 37213
First 5: ['2020/01/IGC-News1-visir_7266555.xml', '2020/01/IGC-News1-visir_7266556.xml', '2020/01/IGC-News1-visir_7266557.xml', '2020/01/IGC-News1-visir_7266558.xml', '2020/01/IGC-News1-visir_7266559.xml']


In [25]:
sample_articles = []

for href in hrefs_year[:500]:
    article_path = wrapper_path.parent / href
    try:
        sample_articles.append(parse_article(article_path))
    except Exception as e:
        print("Failed:", article_path, e)

print("Parsed:", len(sample_articles))
sample_articles[0]

Parsed: 500


{'path': 'data\\IGC-News1\\IGC-News1-22.10.TEI\\visir\\2020\\01\\IGC-News1-visir_7266555.xml',
 'title': '„Dæmigert janúarveður“ næstu daga - Vísir',
 'source': 'Vísir',
 'date': '2020-01-01',
 'keywords': ['Innlent', 'Fréttir'],
 'text': 'Árið byrjar með úrkomu víða um land og er spáð skúrum eða slydduéljum. Þó ert bjart með köflum norðaustantil og heldur þurrt. Hiti hefur fallið frá því á gamlársdag og verður á bilinu 0 til 6 stig á landinu í dag.\nÞá kólnar á morgun og er spáð snjókomu eða éljum víða á landinu, einkum fyrir norðan. Kalt heimskautaloft færist yfir landið og fellur hiti talsvert. Um kvöldið lægir og rofar til á sunnanverðu landinu en spáð frosti í öllum landshlutum á bilinu 3 til 10 stig.\nÞetta kemur fram í hugleiðingum veðurfræðings hjá Veðurstofu Íslands þar sem segir einnig að dagarnir á eftir séu órólegir. Landsmenn megi búast við „dæmigerðu janúarveðri“ með lægðagangi og umhleypingum. Sólin sé hins vegar farin að hækka á lofti og dagarnir farnir að lengjast.\nVe

In [26]:
from collections import Counter

keyword_counts = Counter()

for article in sample_articles:
    keyword_counts.update(article["keywords"])

keyword_counts.most_common(50)

[('Fréttir', 258),
 ('Íþróttir', 154),
 ('Innlent', 143),
 ('Erlendar fréttir', 91),
 ('Fótbolti', 82),
 ('Mannlíf', 38),
 ('Körfubolti', 31),
 ('Viðskipti', 23),
 ('Handbolti', 13),
 ('Viðhorf', 10),
 ('Menning', 8),
 ('Bíó og sjónvarp', 5),
 ('Bílar', 4),
 ('Golf', 4),
 ('Fótbolti - Ísland', 3),
 ('Tónlist', 2),
 ('Veiði', 2),
 ('Tölvuleikir', 1),
 ('Tíska', 1),
 ('Skopmyndir', 1)]

In [27]:
article = sample_articles[0]

for k, v in article.items():
    print(f"{k}: {v if k != 'text' else v[:500]}")
    print()

path: data\IGC-News1\IGC-News1-22.10.TEI\visir\2020\01\IGC-News1-visir_7266555.xml

title: „Dæmigert janúarveður“ næstu daga - Vísir

source: Vísir

date: 2020-01-01

keywords: ['Innlent', 'Fréttir']

text: Árið byrjar með úrkomu víða um land og er spáð skúrum eða slydduéljum. Þó ert bjart með köflum norðaustantil og heldur þurrt. Hiti hefur fallið frá því á gamlársdag og verður á bilinu 0 til 6 stig á landinu í dag.
Þá kólnar á morgun og er spáð snjókomu eða éljum víða á landinu, einkum fyrir norðan. Kalt heimskautaloft færist yfir landið og fellur hiti talsvert. Um kvöldið lægir og rofar til á sunnanverðu landinu en spáð frosti í öllum landshlutum á bilinu 3 til 10 stig.
Þetta kemur fram í huglei

